In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('../')

In [18]:
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

from src.data_extraction.labels import get_label_metadata
from src.data_extraction.attributions import get_attributions_metadata
from src.data_extraction.video_frame import (create_video_frame_metadata_from_label_and_attributions,
                                             create_video_frame_df, 
                                             save_video_frame_metadata_to_csv)
from src.data_extraction.patients import load_patients_metadata_from_csv
from src.create_folders import get_frame_from_video


In [19]:
# Locais de salvamento/acesso dos dados

videos_dir = '..\\data\\videos\\' 
labels_dir = '..\\data\\rotulos\\anotacoes-tecgraf\\'
frames_dir = '..\\data\\frames\\'

### **Coletando e Filtrando dados**

In [20]:
# Coletando Informações dos Rótulos feitos

attribution_metadata_df = get_attributions_metadata(labels_dir)
labels_metadata_df = get_label_metadata(labels_dir)

video_frame_metadata_df = create_video_frame_metadata_from_label_and_attributions(
    label_metadata=labels_metadata_df,
    attributions_metadata=attribution_metadata_df
)
video_frame_metadata_df

Shape of attributions metadata df: (2000, 5)
Shape of label metadata df: (1361, 7)
Shape of labeled frames df: (2000, 9)


,batch,labeler,video_frame,video_id,frame_id,file_path,has_mask,has_points
0,1,AM,v33_f24,33,24,..\data\rotulos\anotacoes-tecgraf\batch-1\AM\v...,True,True
1,1,AM,v54_f127,54,127,..\data\rotulos\anotacoes-tecgraf\batch-1\AM\v...,True,True
2,1,AM,v74_f65,74,65,..\data\rotulos\anotacoes-tecgraf\batch-1\AM\v...,True,True
3,1,BB,v16_f122,16,122,..\data\rotulos\anotacoes-tecgraf\batch-1\BB\v...,True,True
4,1,BB,v66_f166,66,166,..\data\rotulos\anotacoes-tecgraf\batch-1\BB\v...,True,True
...,...,...,...,...,...,...,...,...
1995,5,VR,v70_f137,70,137,NaN,False,False
1996,5,VR,v76_f80,76,80,NaN,False,False
1997,5,VR,v78_f213,78,213,NaN,False,False
1998,5,VR,v89_f118,89,118,NaN,False,False


In [21]:
# Selecionando somente os dados de interesse

video_frame_df = create_video_frame_df(
    videos_dir=videos_dir,
    labels_dir=labels_dir,
    frames_dir=frames_dir,
    batch=None, # Get all batches
    labeler=None, # Get all labelers
    labeler_filter_criteria="union", 
    target='points'
)

video_frame_df

Shape of attributions metadata df: (2000, 5)
Shape of label metadata df: (1361, 7)
Shape of labeled frames df: (2000, 9)


,video_frame,video_id,frame_id,batch,fonte_dados,paciente_id,momento,procedimento,selected_labeler,video_path,frame_path,target_dir
0,v100_f10,100,10,5,video100,115,pos,total,VC,..\data\videos\100.avi,..\data\frames\v100_f10.png,..\data\rotulos\anotacoes-tecgraf\batch-5\VC\v...
1,v100_f12,100,12,2,video100,115,pos,total,BB,..\data\videos\100.avi,..\data\frames\v100_f12.png,..\data\rotulos\anotacoes-tecgraf\batch-2\BB\v...
2,v100_f13,100,13,3,video100,115,pos,total,BB,..\data\videos\100.avi,..\data\frames\v100_f13.png,..\data\rotulos\anotacoes-tecgraf\batch-3\BB\v...
3,v100_f14,100,14,4,video100,115,pos,total,BB,..\data\videos\100.avi,..\data\frames\v100_f14.png,..\data\rotulos\anotacoes-tecgraf\batch-4\BB\v...
4,v100_f15,100,15,5,video100,115,pos,total,VC,..\data\videos\100.avi,..\data\frames\v100_f15.png,..\data\rotulos\anotacoes-tecgraf\batch-5\VC\v...
...,...,...,...,...,...,...,...,...,...,...,...,...
809,v99_f7,99,7,2,video100,114,pos,total,VC,..\data\videos\99.avi,..\data\frames\v99_f7.png,..\data\rotulos\anotacoes-tecgraf\batch-2\VC\v...
810,v9_f118,9,118,5,video100,24,pos,total,VC,..\data\videos\9.avi,..\data\frames\v9_f118.png,..\data\rotulos\anotacoes-tecgraf\batch-5\VC\v...
811,v9_f40,9,40,2,video100,24,pos,total,VC,..\data\videos\9.avi,..\data\frames\v9_f40.png,..\data\rotulos\anotacoes-tecgraf\batch-2\VC\v...
812,v9_f50,9,50,3,video100,24,pos,total,BB,..\data\videos\9.avi,..\data\frames\v9_f50.png,..\data\rotulos\anotacoes-tecgraf\batch-3\BB\v...


In [34]:
# Salvando dados de interesse
save_video_frame_metadata_to_csv(video_frame_df, filename='video_frame_metadata.csv', output_dir='../data/metadados/')

Video frame Metadata saved to ../data/metadados/video_frame_metadata.csv


In [23]:
# Coleta Informações dos Vídeos Possuídos 
patients_df = load_patients_metadata_from_csv()

### **Salvando Frames de Interesse para uso futuro**

In [24]:
# Salva frames de que foram reotulados

def check_empty_folder(folder_path):
    if os.listdir(folder_path):  # Se a pasta NÃO estiver vazia
        raise RuntimeError(f"A pasta '{folder_path}' não está vazia! Esvazie caso queira um reprocessamento.")
    else:
        print(f"A pasta '{folder_path}' está vazia.")


check_empty_folder(frames_dir)

for i in tqdm(range(len(video_frame_df)), desc = "Frames Salvos:"):
    df_aux = video_frame_df.loc[i]
    frame_id = int(df_aux["frame_id"])
    video_path = df_aux["video_path"]
    frame_path = df_aux["frame_path"]

    frame = get_frame_from_video(video_path, frame_id)
    plt.imsave(frame_path, frame)


RuntimeError: A pasta '..\data\frames\' não está vazia! Esvazie caso queira um reprocessamento.

### **Breve analise dos dados rotulados filtrados**

In [25]:
# Quantidade de frames rotulados por vídeo

fig = px.histogram(
    video_frame_df.video_id,
    title='Quantidade de frames rotulados por vídeo',
    histnorm='probability density',
    text_auto=True
)

fig.update_layout(
    bargap=0.2,
    yaxis_tickformat=',.0%',
    width=800,
    height=500
)

In [26]:
# Número de vídeos com a mesma quantidade de frames rotulados

fig = px.histogram(
    video_frame_df.video_id.value_counts(), 
    title='Número de vídeos com a mesma quantidade de frames rotulados',
    histnorm='probability density',
    text_auto=True
)


fig.update_layout(
    bargap=0.2,
    yaxis_tickformat=',.0%',
    width=800,
    height=500
)

In [27]:
# Quantidade de frames rotulados por rotuladores

fig = px.histogram(
    video_frame_df.selected_labeler, 
    title='Quantidade de frames rotulados por labeler',
    histnorm='probability density',
    text_auto=True
)

fig.update_xaxes(categoryorder='total descending')

fig.update_layout(
    bargap=0.2,
    yaxis_tickformat=',.0%',
    width=800,
    height=500, 
)

In [28]:
# Frequência dos momentos (pre/pos) 

fig = px.histogram(
    patients_df,
    x='momento',
    title='Distribuição de vídeos por momento',
    text_auto=True,
    category_orders={'momento': ['pre', 'pos']}
)

fig.update_layout(
    bargap=0.2,
    width=600,
    height=500
)

In [ ]:
# Distribuição de vídeos por tipo de exame

fig = px.histogram(
    patients_df[patients_df.momento == 'pos'],
    x='procedimento',
    title='Distribuição de vídeos por tipo de exame',
    text_auto=True,
)

fig.update_layout(
    bargap=0.2,
    width=800,
    height=500
)
fig.show()

print(patients_df[patients_df.momento == 'pos'].procedimento.value_counts(normalize=True))


procedimento
total            0.677419
chep             0.174194
microcirurgia    0.058065
radioterapia     0.038710
near_total       0.032258
desconhecido     0.019355
Name: proportion, dtype: float64


In [33]:
# Plot a distribuição dos pacientes
distribuicao_pacientes = (
    video_frame_df
    .paciente_id
    .value_counts()
    .to_frame()
    .reset_index()
)
distribuicao_pacientes.columns = ['paciente_id', 'qtt_frames']
# Change paciente_id to category
distribuicao_pacientes['paciente_id'] = distribuicao_pacientes['paciente_id'].astype('str')
#distribuicao_pacientes = distribuicao_pacientes.head(20)

fig = px.histogram(
    distribuicao_pacientes,
    y='paciente_id',
    x='qtt_frames',
    title='Distribuição da quantidade de frames por paciente',
    # histnorm='probability density',
    text_auto=True,
    orientation='h'
)

# Sort bars in descending order
fig.update_yaxes(categoryorder='total ascending')

fig.update_layout(
    bargap=0.2,
    width=800,
    height=500
)

fig.show()